# IMPORTACIONES

In [22]:
from google.colab import drive
import pandas as pd
import json

# CONEXIÓN DE DRIVE

In [23]:
# Montamos Drive en nuestro espacio de trabajo
drive.mount('/content/drive')

# Cargamos los datos en un Dataframe
data = pd.read_csv('/content/drive/MyDrive/DIET-IA/dataset/13k-recipes.csv')

# Visualizamos las primeras filas para verificar alérgenos e ingredientes
print(data.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   Unnamed: 0                                              Title  \
0           0  Miso-Butter Roast Chicken With Acorn Squash Pa...   
1           1                    Crispy Salt and Pepper Potatoes   
2           2                        Thanksgiving Mac and Cheese   
3           3                 Italian Sausage and Bread Stuffing   
4           4                                       Newton's Law   

                                         Ingredients  \
0  ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...   
1  ['2 large egg whites', '1 pound new potatoes (...   
2  ['1 cup evaporated milk', '1 cup whole milk', ...   
3  ['1 (¾- to 1-pound) round Italian loaf, cut in...   
4  ['1 teaspoon dark brown sugar', '1 teaspoon ho...   

                                        Instructions  \
0  Pat chicken dry with paper towels, season all ...   
1  Preheat ov

In [45]:
import re
import ast  

data['Ingredients_list'] = data['Ingredients'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip() else []
)

def limpiar(ing_list):
    if not ing_list: return 'unknown'
    text = ' '.join(str(ing).lower() for ing in ing_list)
    text = re.sub(r'\b\d+(?:[./]\d+)?\s*(cups?|tsp|tbsp|lbs?|oz|g|ml)\b', '', text)
    palabras = re.findall(r'\b[a-z]{3,}\b', text)
    return ' '.join(dict.fromkeys(palabras))

data['ingredients_clean'] = data['Ingredients_list'].apply(limpiar)

print("✅ ¡PERFECTO! Primeras 5:")
print(data[['Title', 'ingredients_clean']].head().to_string(index=False))


✅ ¡PERFECTO! Primeras 5:
                                                 Title                                                                                                                                                                                                                                                                                                                                                                                                                 ingredients_clean
Miso-Butter Roast Chicken With Acorn Squash Panzanella whole chicken tsp kosher salt divided plus more small acorn squash about total finely chopped sage rosemary unsalted butter melted room temperature ground allspice pinch crushed red pepper flakes freshly black loaf good quality sturdy white bread torn into pieces cups medium apples such gala pink lady cored cut extra virgin olive oil onion thinly sliced apple cider vinegar miso cup all purpose flour dry wine broth
                       Crispy

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=15000, min_df=1, ngram_range=(1,2))
X = vectorizer.fit_transform(data['ingredients_clean'].fillna(''))
print(f"✅ X: {X.shape} | Vocab: {len(vectorizer.vocabulary_)}")


✅ X: (13501, 15000) | Vocab: 15000


In [48]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn.fit(X)
print("✅ Entrenado")


✅ Entrenado


In [49]:
import pickle
modelo = {
    'vectorizer': vectorizer, 'knn': knn, 'X': X,
    'data': data[['Title', 'Ingredients_list', 'Instructions']].reset_index(drop=True),
    'limpiar': limpiar
}
with open('/content/modelo_FINAL.pkl', 'wb') as f:
    pickle.dump(modelo, f)
print("✅ Listo /content/modelo_FINAL.pkl")


✅ Listo /content/modelo_FINAL.pkl


In [50]:
def recomendar(texto, min_sim=0.15, k=10):
    user_list = texto.split(',')
    user_clean = modelo['limpiar'](user_list)
    q = modelo['vectorizer'].transform([user_clean])
    dist, idx = modelo['knn'].kneighbors(q, k)
    
    print(f"\n🔥 TOP recetas para '{texto}' (procesado: {user_clean[:70]})")
    matches = []
    for d, idr in zip(dist[0], idx[0]):
        sim = 1 - d
        if sim >= min_sim:
            row = modelo['data'].iloc[idr]
            matches.append((sim, row['Title'], str(row['Ingredients_list'])[:120]))
    
    for i, (sim, title, ing) in enumerate(sorted(matches, reverse=True)[:5]):
        print(f"{i+1}. **{title[:55]}** ← sim:{sim:.3f}")
        print(f"   {ing}...")
    print(f"\n{len(matches)} matches >{min_sim}")


In [51]:
recomendar("chicken lemon garlic rosemary")



🔥 TOP recetas para 'chicken lemon garlic rosemary' (procesado: chicken lemon garlic rosemary)
1. **Arugula Salsa Verde** ← sim:0.236
   ['2 plum tomatoes, finely chopped (optional)', 'Finely grated zest of 1 small lemon', '1 garlic clove, finely chopped', ...
2. **White Pesto Pasta** ← sim:0.215
   ['½ cup walnuts', 'Kosher salt', '½ cup fresh ricotta', 'Zest of 1 lemon', '1 garlic clove, finely grated', '2 tsp. fine...
3. **Instant Pot Lemon Chicken With Garlic and Olives** ← sim:0.197
   ['1 tablespoon extra virgin olive oil, plus more as needed', '2 pounds bone-in, skin-on chicken thighs, patted dry (4 to...
4. **Grilled Lemon-Garlic Chicken with Leeks and Potatoes** ← sim:0.189
   ['1 lemon', '3 garlic cloves, finely grated', '1 1/2 teaspoons kosher salt', '3/4 teaspoon smoked paprika', '3 tablespoo...
5. **Fancy and Beautiful Tomato Salad** ← sim:0.185
   ['1½ lb. heirloom tomatoes (about 3 medium), sliced into 8–12 wedges, depending on size', '12 oz. mixed cherry tomatoes ...

1